### Merge script

In [24]:
import pandas as pd

In [25]:
iSUPER = pd.read_csv("Filtered dataset/isuper_5min_full.csv")
Weather_underground  = pd.read_csv("Filtered dataset/Weather_Underground_5min_East_Boston_full.csv")
ASOS = pd.read_csv("Filtered dataset/ASOS_visibility_5min_clean.csv")

In [26]:
iSUPER["timestamp_utc"] = pd.to_datetime(iSUPER["timestamp_utc"], utc=True)
Weather_underground["timestamp_utc"]  = pd.to_datetime(Weather_underground["timestamp_utc"],  utc=True)
ASOS["timestamp_utc"] = pd.to_datetime(ASOS["timestamp_utc"], utc=True, errors="coerce")

In [27]:
iSUPER = iSUPER.sort_values("timestamp_utc")
Weather_underground  = Weather_underground.sort_values("timestamp_utc")
ASOS = ASOS.sort_values("timestamp_utc")

display(iSUPER.head(), Weather_underground.head(), ASOS.head())

,timestamp_utc,rh_sensor,temp_sensor,pm1,pm25,pm10
0,2023-02-14 20:05:00+00:00,18.550000,16.050000,2.0505,2.507000,19.530500
1,2023-02-14 20:10:00+00:00,18.820000,15.840000,2.2178,2.529600,8.615400
2,2023-02-14 20:15:00+00:00,17.740000,15.780000,2.2174,2.558800,18.074600
3,2023-02-14 20:20:00+00:00,17.780000,15.600000,1.8958,2.198200,13.013000
4,2023-02-14 20:25:00+00:00,18.214286,15.671429,1.9210,2.382143,24.699714


,timestamp_utc,Temperature_C,Dew_Point_C,Humidity_%,Speed_kmh,Pressure_hPa,Precip_Rate_mm,Precip_Accum_mm,Wind,Dew/Temp fog?,RH fog?
0,2023-07-10 00:15:00+00:00,21.11,19.56,91.0,10.46,1011.18,0.0,0.0,East,1.0,0.0
1,2023-07-10 00:20:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-07-10 00:25:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-07-10 00:30:00+00:00,21.28,19.56,90.0,5.47,1011.18,0.0,0.0,SSE,1.0,0.0
4,2023-07-10 00:35:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,timestamp_utc,tmpc_ASOS,dewpoint_ASOS,vis1_coeff,vis2_coeff,vis3_coeff,ext_coeff_mean,vis1_km,vis2_km,vis3_km,visibility_km_mean
0,2023-01-01 05:00:00+00:00,12.22,12.22,0.269,1.493,0.279,0.680333,17.948030,3.233771,17.304731,12.828844
1,2023-01-01 05:05:00+00:00,12.22,12.22,0.173,0.283,0.226,0.227333,27.907630,17.060141,21.362920,22.110231
2,2023-01-01 05:10:00+00:00,12.22,12.22,0.267,0.326,0.252,0.281667,18.082472,14.809877,19.158810,17.350386
3,2023-01-01 05:15:00+00:00,12.22,12.22,0.380,0.414,0.424,0.406000,12.705316,11.661884,11.386840,11.918013
4,2023-01-01 05:20:00+00:00,12.22,12.22,0.231,0.359,0.340,0.310000,20.900519,13.448524,14.200059,16.183034


In [28]:
df_merged = iSUPER.merge(
    Weather_underground,
    on="timestamp_utc",
    how="inner",
    suffixes=("_che", "_wu")
)

display(df_merged.head()) #df_merged.head()
df_merged.shape

,timestamp_utc,rh_sensor,temp_sensor,pm1,pm25,pm10,Temperature_C,Dew_Point_C,Humidity_%,Speed_kmh,Pressure_hPa,Precip_Rate_mm,Precip_Accum_mm,Wind,Dew/Temp fog?,RH fog?
0,2023-07-10 00:15:00+00:00,82.54,23.2,2.6268,3.1144,7.9832,21.11,19.56,91.0,10.46,1011.18,0.0,0.0,East,1.0,0.0
1,2023-07-10 00:20:00+00:00,82.72,23.2,2.5710,3.0144,6.8504,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-07-10 00:25:00+00:00,82.72,23.2,2.6030,3.0028,6.1358,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-07-10 00:30:00+00:00,82.62,23.2,2.5456,3.0116,12.2262,21.28,19.56,90.0,5.47,1011.18,0.0,0.0,SSE,1.0,0.0
4,2023-07-10 00:35:00+00:00,82.76,23.2,2.9380,3.5138,8.6174,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


(247101, 16)

In [29]:
# Attach ASOS visibility and only some related variables
df_merged = df_merged.merge(
    ASOS[[
            "timestamp_utc",
            "tmpc_ASOS",
            "dewpoint_ASOS",
            "visibility_km_mean",
        ]
    ],
    on="timestamp_utc",
    how="left",
)

In [30]:
# round all numeric columns
numeric_cols = df_merged.select_dtypes(include="number").columns
df_merged[numeric_cols] = df_merged[numeric_cols].round(2)

In [31]:
df_merged.head()

,timestamp_utc,rh_sensor,temp_sensor,pm1,pm25,pm10,Temperature_C,Dew_Point_C,Humidity_%,Speed_kmh,Pressure_hPa,Precip_Rate_mm,Precip_Accum_mm,Wind,Dew/Temp fog?,RH fog?,tmpc_ASOS,dewpoint_ASOS,visibility_km_mean
0,2023-07-10 00:15:00+00:00,82.54,23.2,2.63,3.11,7.98,21.11,19.56,91.0,10.46,1011.18,0.0,0.0,East,1.0,0.0,21.11,20.0,52.31
1,2023-07-10 00:20:00+00:00,82.72,23.2,2.57,3.01,6.85,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.11,20.0,50.39
2,2023-07-10 00:25:00+00:00,82.72,23.2,2.60,3.00,6.14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.11,20.0,51.90
3,2023-07-10 00:30:00+00:00,82.62,23.2,2.55,3.01,12.23,21.28,19.56,90.0,5.47,1011.18,0.0,0.0,SSE,1.0,0.0,21.11,20.0,52.27
4,2023-07-10 00:35:00+00:00,82.76,23.2,2.94,3.51,8.62,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.11,20.0,53.01


### Create labels

(add in the NOAA source to support this)

Base on this paper:

https://agupubs.onlinelibrary.wiley.com/doi/full/10.1029/2018jd029389

Section 2.3: "A day is defined as a fog day when RH ≥ 90% and VIS ≤ 1 km at any time of the day or a haze day when RH < 90% and VIS ≤ 10 km at 14:00 LT. Other low-visibility weather phenomena, such as rain, snow, sand, and dust storm are excluded. "

In [32]:
# drop old flags
df_merged = df_merged.drop(columns=["Dew/Temp fog?", "RH fog?"])
df_merged.head(3)

,timestamp_utc,rh_sensor,temp_sensor,pm1,pm25,pm10,Temperature_C,Dew_Point_C,Humidity_%,Speed_kmh,Pressure_hPa,Precip_Rate_mm,Precip_Accum_mm,Wind,tmpc_ASOS,dewpoint_ASOS,visibility_km_mean
0,2023-07-10 00:15:00+00:00,82.54,23.2,2.63,3.11,7.98,21.11,19.56,91.0,10.46,1011.18,0.0,0.0,East,21.11,20.0,52.31
1,2023-07-10 00:20:00+00:00,82.72,23.2,2.57,3.01,6.85,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.11,20.0,50.39
2,2023-07-10 00:25:00+00:00,82.72,23.2,2.60,3.00,6.14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.11,20.0,51.90


In [33]:
df_merged["fog_label"] = (
    (df_merged["Humidity_%"] > 90.0) &
    (df_merged["visibility_km_mean"] < 1.0)
).astype("Int64")

In [34]:
df_merged["fog_label"].value_counts()

fog_label
0    245982
1      1119
Name: count, dtype: Int64

In [35]:
df_merged.loc[df_merged["fog_label"] == 1].head(8)

,timestamp_utc,rh_sensor,temp_sensor,pm1,pm25,pm10,Temperature_C,Dew_Point_C,Humidity_%,Speed_kmh,Pressure_hPa,Precip_Rate_mm,Precip_Accum_mm,Wind,tmpc_ASOS,dewpoint_ASOS,visibility_km_mean,fog_label
98,2023-07-10 08:25:00+00:00,89.80,21.60,1.60,1.76,173.99,20.00,19.39,96.0,3.54,1009.48,0.0,0.0,ESE,20.00,20.0,0.89,1
100,2023-07-10 08:35:00+00:00,89.94,21.60,1.78,2.09,182.25,20.00,19.39,96.0,2.57,1009.48,0.0,0.0,ESE,20.00,20.0,0.98,1
101,2023-07-10 08:40:00+00:00,90.06,21.60,1.58,1.85,243.54,19.94,19.39,96.0,4.67,1009.48,0.0,0.0,ESE,20.00,20.0,0.81,1
102,2023-07-10 08:45:00+00:00,90.00,21.60,1.46,1.84,327.48,19.94,19.33,96.0,3.54,1009.14,0.0,0.0,East,20.00,20.0,0.68,1
103,2023-07-10 08:50:00+00:00,90.10,21.58,1.66,2.08,300.44,20.00,19.39,96.0,4.67,1009.14,0.0,0.0,ESE,20.00,20.0,0.97,1
112,2023-07-10 09:35:00+00:00,90.00,21.50,1.22,1.53,275.58,19.78,19.39,97.0,5.47,1009.48,0.0,0.0,ESE,20.00,20.0,0.94,1
113,2023-07-10 09:40:00+00:00,90.12,21.50,1.38,1.67,273.93,19.83,19.39,97.0,3.70,1009.48,0.0,0.0,SE,20.00,20.0,0.91,1
114,2023-07-10 09:45:00+00:00,90.08,21.46,1.50,1.86,317.05,19.78,19.39,97.0,7.24,1009.48,0.0,0.0,ESE,19.44,20.0,0.71,1


### Output

In [36]:
df_merged.to_csv("Filtered dataset/Training_data.csv", index=False)

### Result exploration

In [37]:
# Check if all 3 temps are going up or down similarly or not


In [38]:
# when fog is 0 for both Dew/temp and RH, what are the PM values?
# change this because old flags are dropped
print("PM1 mean: ",df_merged.loc[(df_merged["Dew/Temp fog?"] == 0) & (df_merged["RH fog?"] == 0), "pm1"].mean())
print("PM2.5 mean: ",df_merged.loc[(df_merged["Dew/Temp fog?"] == 0) & (df_merged["RH fog?"] == 0), "pm25"].mean())
print("PM10 mean: ",df_merged.loc[(df_merged["Dew/Temp fog?"] == 0) & (df_merged["RH fog?"] == 0), "pm10"].mean())

print("PM10 median: ",df_merged.loc[(df_merged["Dew/Temp fog?"] == 0) & (df_merged["RH fog?"] == 0), "pm10"].median())

KeyError: 'Dew/Temp fog?'

In [ ]:
# when fog is 1 for both Dew/temp and RH, what are the PM values?
print("PM1 mean: ",df_merged.loc[(df_merged["Dew/Temp fog?"] == 1) & (df_merged["RH fog?"] == 1), "pm1"].mean())
print("PM2.5 mean: ",df_merged.loc[(df_merged["Dew/Temp fog?"] == 1) & (df_merged["RH fog?"] == 1), "pm25"].mean())
print("PM10 mean: ",df_merged.loc[(df_merged["Dew/Temp fog?"] == 1) & (df_merged["RH fog?"] == 1), "pm10"].mean())

# higher PM10 mean compared to no fog, however median shows that it is not much higher in fact only the outliners
print("PM10 median: ",df_merged.loc[(df_merged["Dew/Temp fog?"] == 1) & (df_merged["RH fog?"] == 1), "pm10"].median())

PM1 mean:  4.568538379471054
PM2.5 mean:  6.659879836037958
PM10 mean:  74.25151047223315
PM10 median:  17.34


In [ ]:
# when fog is 1 for both Dew/temp and RH, what are the PM values?
df_merged.loc[(df_merged["Dew/Temp fog?"] == 1) & (df_merged["RH fog?"] == 1) & (df_merged["Precip_Rate_mm"] == 0), "pm10"].mean()

90.39462111223145